# AE3 · Transformación y enriquecimiento de ventas de cafetería
**Autor:** Jose David Osorio Gallego  
**Asignatura:** Proyecto Integrado III – Analítica de Datos · **Grupo:** PREICA2602B020038  
**Docente:** Andrés Felipe Callejas · Institución Universitaria Digital de Antioquia

## Necesidad y continuidad con AE2
La cafetería es un escenario hipotético. Se necesita describir cómo se distribuyen las unidades y los importes entre productos en 2023. Se conserva el dataset completo y la pregunta central de AE2: **¿Cómo se distribuyen las unidades vendidas y el importe total registrado entre los tipos de producto en las transacciones sintéticas de cafetería de 2023?**

P1 y P2 descomponen esa pregunta; P3 desarrolla el complemento mensual ya previsto en AE2. P4 cuantifica la modalidad desconocida para hacer explícita la limitación de calidad identificada en AE2. No reemplaza la pregunta central ni introduce una fuente nueva.

Datos **sintéticos**, con imputaciones previas del publicador: no corresponden a una empresa real. No hay costos, inventarios ni marcas por fila de los valores imputados. Los resultados describen el archivo limpio, no ventas observadas sin intervención.

**Ejecución:** ejecutar todas las celdas desde este directorio o desde la raíz del repositorio. Se utiliza la copia local inalterada del CSV de AE2, con revisión y SHA256 verificados. No requiere Internet para analizar y exportar. El código de transformación completo está en `AE3/src/analisis.py`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

# Localización relativa: admite Jupyter desde raíz, AE3 o AE3/notebooks.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'AE3/src/analisis.py').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Abrir el notebook dentro del repositorio')
sys.path.insert(0, str(RAIZ / 'AE3/src'))
from analisis import (PREGUNTAS, REVISION, URL, cargar, transformar, resultados,
                      interpretaciones, diccionario, exportar, resumen_descriptivo)
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 100)
display(PREGUNTAS)

,pregunta_id,pregunta,variables_originales,nivel,resultado_esperado,archivo
0,P1,¿Cómo se distribuyen las unidades vendidas entre productos en 2023?,"Item, Quantity, Transaction Date",Producto,Unidades y participación porcentual,p1_unidades_producto.csv
1,P2,¿Cómo se distribuye el importe registrado entre productos en 2023?,"Item, Total Spent, Transaction Date",Producto,USD y participación porcentual,p2_importe_producto.csv
2,P3,¿Cómo varían los importes registrados por mes durante 2023?,"Transaction Date, Item, Total Spent, Quantity",Mes; complemento mes y producto,"USD, unidades y variación mensual porcentual",p3_ventas_mes.csv
3,P4,¿Qué proporción de registros e importes tiene modalidad desconocida?,"Location, Total Spent",Modalidad,"Registros, USD y porcentajes de cobertura",p4_cobertura_modalidad.csv


## 1 · Origen y versión sin modificar
Fuente: [Aditya Suyal · Cafe Sales Dataset Cleaned](https://huggingface.co/datasets/adityasuyal/cafe-sales-cleaned), revisión `f4e6dfa1fe791ba84807921c949bd84f02b4bc69`, archivo `cleaned_cafe_sales.csv`. La ficha fuente se conserva en `AE3/datos/originales/FICHA_FUENTE.md`; declara licencia MIT y moneda USD. Se mantiene atribución en `AVISO_FUENTE.txt`.

El publicador pasó de 10.000 a 9.540 filas al retirar fechas no válidas; además imputó categorías con la moda y cantidades/precios con medianas, y recalculó `Total Spent`. **AE3 no repite ni atribuye a este trabajo esa limpieza previa.** La concordancia aritmética no constituye una validación independiente de ventas reales.

In [2]:
original, ruta_origen = cargar(RAIZ)
original_respaldo = original.copy(deep=True)
print('Revisión:', REVISION)
print('Origen fijado:', URL)
print('Dimensiones originales:', original.shape)
display(original.head())
display(pd.DataFrame({'tipo': original.dtypes.astype(str), 'nulos': original.isna().sum()}))
original.info()

Revisión: f4e6dfa1fe791ba84807921c949bd84f02b4bc69
Origen fijado: https://huggingface.co/datasets/adityasuyal/cafe-sales-cleaned/resolve/f4e6dfa1fe791ba84807921c949bd84f02b4bc69/cleaned_cafe_sales.csv
Dimensiones originales: (9540, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9540 entries, 0 to 9539
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    9540 non-null   object 
 1   Item              9540 non-null   object 
 2   Quantity          9540 non-null   float64
 3   Price Per Unit    9540 non-null   float64
 4   Total Spent       9540 non-null   float64
 5   Payment Method    9540 non-null   object 
 6   Location          9540 non-null   object 
 7   Transaction Date  9540 non-null   object 
dtypes: float64(3), object(5)
memory usage: 596.4+ KB


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Unknown,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


,tipo,nulos
Transaction ID,object,0
Item,object,0
Quantity,float64,0
Price Per Unit,float64,0
Total Spent,float64,0
Payment Method,object,0
Location,object,0
Transaction Date,object,0


## 2 · Transformación y reglas de calidad
Se renombra el esquema en español, se convierte la fecha con formato exacto y la cantidad a entero solo tras comprobar que no tiene fracciones. Se tipan las categorías conservando sus etiquetas. Los precios e importes siguen en USD. Los campos de pago e identificador se conservan para trazabilidad, aunque no se utilicen como dimensiones de las preguntas principales.

No se elimina ninguna fila ni columna y no se imputan valores. `Unknown` sigue siendo una categoría explícita. Los controles detienen la ejecución si aparecen nulos, fechas inválidas, duplicados, valores fuera del catálogo, cantidades fraccionarias o desajustes de importe; una anomalía nueva requiere revisar el criterio, no borrarla silenciosamente. El registro de operaciones distingue filas transformadas de problemas detectados.

In [3]:
df, calidad, comparacion, cambios = transformar(original)
pd.testing.assert_frame_equal(original, original_respaldo)
display(cambios)
display(calidad)
display(pd.DataFrame({'etapa':['Original AE2','Preparado AE3'],
                      'filas':[len(original),len(df)],
                      'columnas':[original.shape[1],df.shape[1]],
                      'nulos':[int(original.isna().sum().sum()),int(df.isna().sum().sum())]}))
display(comparacion)

,operacion,filas_afectadas,justificacion
0,Renombrar 8 columnas,9540,Etiquetas en español; no cambia valores
1,Convertir fecha de texto a datetime,9540,Permite delimitar el año y agrupar meses
2,Convertir cantidad de float64 a Int64,9540,Todas las cantidades son enteras; sin redondeo
3,"Tipar producto, pago y modalidad como category",9540,Tres campos por fila; se conservan etiquetas y Unknown
4,Recortar espacios externos,0,Sin correcciones si el conteo es cero
5,Eliminar filas,0,No se detectan problemas que justifiquen eliminación
6,Imputar faltantes,0,El CSV limpio no contiene nulos; Unknown se conserva
7,Cambiar unidad monetaria,0,Se mantiene USD; no se convierte a COP
8,Añadir cinco variables derivadas,9540,"Año, mes, importe calculado, diferencia y cobertura"


,control,registros_o_valores_detectados,criterio,tratamiento
0,Valores nulos en origen,0,No imputar automáticamente,Conservar; no se detectó problema
1,Espacios externos en id_transaccion,0,Recortar espacios sin alterar significado,Sin cambios de contenido
2,Espacios externos en producto,0,Recortar espacios sin alterar significado,Sin cambios de contenido
3,Espacios externos en medio_pago,0,Recortar espacios sin alterar significado,Sin cambios de contenido
4,Espacios externos en modalidad,0,Recortar espacios sin alterar significado,Sin cambios de contenido
5,Filas duplicadas,0,No borrar sin identificar su causa,Conservar; no se detectó problema
6,Identificadores duplicados,0,Se exige identificador único,Conservar; no se detectó problema
7,Fechas inválidas,0,Formato exacto AAAA-MM-DD; detener si falla,Conservar; no se detectó problema
8,Registros fuera de 2023,0,Alcance anual definido en AE2,Conservar; no se detectó problema
9,No positivos o no finitos en cantidad,0,"Cantidad, precio e importe deben ser positivos",Conservar; no se detectó problema


,etapa,filas,columnas,nulos
0,Original AE2,9540,8,0
1,Preparado AE3,9540,13,0


,columna_original,columna_final,tipo_antes,tipo_despues,nulos_antes,nulos_despues
0,Transaction ID,id_transaccion,object,string,0,0
1,Item,producto,object,category,0,0
2,Quantity,cantidad,float64,Int64,0,0
3,Price Per Unit,precio_unitario_usd,float64,float64,0,0
4,Total Spent,importe_usd,float64,float64,0,0
5,Payment Method,medio_pago,object,category,0,0
6,Location,modalidad,object,category,0,0
7,Transaction Date,fecha,object,datetime64[ns],0,0


## 3 · Enriquecimiento y diccionario actualizado
Se incorporan cinco variables: `anio = fecha.dt.year`, `mes = fecha.dt.to_period('M')`, `importe_calculado_usd = round(cantidad × precio_unitario_usd, 2)`, `diferencia_importe_usd = round(importe_usd − importe_calculado_usd, 2)` y `modalidad_conocida = modalidad != 'Unknown'`.

Año y mes permiten delimitar y agrupar el periodo; importe calculado y diferencia permiten conciliar las respuestas monetarias sin reemplazar el dato fuente; modalidad conocida permite medir cobertura. No se realizan uniones con otras tablas. Por ello no hay claves de unión ni riesgo de multiplicar filas por un cruce. Los nombres originales y su correspondencia quedan en la comparación de esquema.

In [4]:
display(diccionario(df))
display(df.head())
print('Nulos finales por columna:')
display(df.isna().sum().to_frame('nulos'))

Nulos finales por columna:


,variable,significado,tipo,unidad,origen_o_regla
0,id_transaccion,Identificador del registro original,string,Sin unidad,Transaction ID; conservar unicidad
1,producto,Producto del menú,category,Categoría,Item; mismas ocho etiquetas
2,cantidad,Unidades del registro,Int64,Unidades,Quantity; conversión exacta a entero
3,precio_unitario_usd,Precio por unidad,float64,USD/unidad,Price Per Unit; sin conversión monetaria
4,importe_usd,Importe del registro en el CSV limpio,float64,USD,Total Spent; se conserva
5,medio_pago,Medio de pago registrado,category,Categoría,Payment Method; etiquetas originales
6,modalidad,Modalidad declarada o desconocida,category,Categoría,Location; Unknown se mantiene
7,fecha,Fecha de la transacción,datetime64[ns],AAAA-MM-DD,Transaction Date; pd.to_datetime con formato exacto
8,anio,Año de cobertura,int16,Año,fecha.dt.year; verifica alcance 2023
9,mes,Periodo de agregación,string,AAAA-MM,"fecha.dt.to_period(""M""); responde P3"


,id_transaccion,producto,cantidad,precio_unitario_usd,importe_usd,medio_pago,modalidad,fecha,anio,mes,importe_calculado_usd,diferencia_importe_usd,modalidad_conocida
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,2023,2023-09,4.0,0.0,True
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,2023,2023-05,12.0,0.0,True
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19,2023,2023-07,4.0,0.0,True
3,TXN_7034554,Salad,2,5.0,10.0,Digital Wallet,Unknown,2023-04-27,2023,2023-04,10.0,0.0,False
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,2023,2023-06,4.0,0.0,True


,nulos
id_transaccion,0
producto,0
cantidad,0
precio_unitario_usd,0
importe_usd,0
medio_pago,0
modalidad,0
fecha,0
anio,0
mes,0


## 4 · Respuestas sustentadas en tablas
Las participaciones se calculan con el total anual completo como denominador. Se conservan valores sin redondear en memoria y se exportan con diez decimales; en las tablas de lectura se muestran dos. Una suma de porcentajes mostrados puede diferir de 100 por redondeo. `registros` cuenta filas, no clientes ni pedidos completos. La variación mensual es `(importe_mes / importe_mes_anterior − 1) × 100`; en enero es no disponible por falta de diciembre de 2022.

In [5]:
tablas, validaciones = resultados(df)
respuestas = interpretaciones(df, tablas)
print('Total de unidades:', int(df.cantidad.sum()))
print(f'Total de importe: USD {df.importe_usd.sum():,.2f}')

Total de unidades: 28850
Total de importe: USD 85,246.50


### P1 · ¿Cómo se distribuyen las unidades vendidas entre productos en 2023?

In [6]:
display(tablas['p1_unidades_producto'].round(2))
display(Markdown(respuestas['P1']))

,producto,registros,unidades,participacion_unidades_pct
0,Juice,2051,6141,21.29
1,Coffee,1123,3425,11.87
2,Salad,1099,3314,11.49
3,Cake,1082,3278,11.36
4,Sandwich,1075,3266,11.32
5,Smoothie,1048,3208,11.12
6,Tea,1027,3125,10.83
7,Cookie,1035,3093,10.72


Juice concentra el mayor número de unidades: 6,141, equivalentes al 21.29% de 28,850 unidades anuales. Esta distribución permite comparar el volumen del menú en el escenario académico. No representa demanda real: el publicador imputó productos con la moda y cantidades con la mediana; el archivo limpio no identifica qué filas fueron imputadas.

### P2 · ¿Cómo se distribuye el importe registrado entre productos en 2023?

In [7]:
display(tablas['p2_importe_producto'].round(2))
display(Markdown(respuestas['P2']))

,producto,registros,importe_usd,participacion_importe_pct
0,Juice,2051,18249.0,21.41
1,Salad,1099,16198.0,19.00
2,Sandwich,1075,12892.0,15.12
3,Smoothie,1048,12643.0,14.83
4,Cake,1082,9834.0,11.54
5,Coffee,1123,6999.0,8.21
6,Tea,1027,4984.5,5.85
7,Cookie,1035,3447.0,4.04


Juice aporta el mayor importe: USD 18,249.00, el 21.41% de USD 85,246.50. El liderazgo por importe y por unidades puede diferir porque interviene el precio unitario. La tabla sirve para comparar composición de importes, sin concluir rentabilidad. El total fue recalculado por el publicador a partir de cantidades y precios, algunos imputados.

### P3 · ¿Cómo varían los importes registrados por mes durante 2023?

In [8]:
display(tablas['p3_ventas_mes'].round(2))
display(Markdown(respuestas['P3']))

,mes,registros,unidades,importe_usd,participacion_importe_pct,variacion_mensual_pct
0,2023-01,818,2463,7258.0,8.51,NaN
1,2023-02,727,2248,6673.0,7.83,-8.06
2,2023-03,827,2480,7215.5,8.46,8.13
3,2023-04,774,2387,7193.5,8.44,-0.30
4,2023-05,777,2342,6978.5,8.19,-2.99
5,2023-06,818,2471,7417.5,8.70,6.29
6,2023-07,791,2361,6927.5,8.13,-6.61
7,2023-08,803,2405,7101.0,8.33,2.50
8,2023-09,788,2363,6886.0,8.08,-3.03
9,2023-10,838,2535,7353.5,8.63,6.79


El mayor importe mensual corresponde a 2023-06, con USD 7,417.50; el menor, a 2023-02, con USD 6,673.00. El mayor aumento frente al mes previo ocurre en 2023-03 (8.13%) y la mayor caída en 2023-02 (-8.06%). Enero no tiene variación calculable por falta de diciembre de 2022. Esta tabla permite planear la comparación temporal de AE4. Un solo año sintético no demuestra estacionalidad ni permite atribuir causas; los meses tienen distinta duración.

### P4 · ¿Qué proporción de registros e importes tiene modalidad desconocida?

In [9]:
display(tablas['p4_cobertura_modalidad'].round(2))
display(Markdown(respuestas['P4']))

,modalidad,registros,importe_usd,participacion_registros_pct,participacion_importe_pct
0,In-store,2872,25906.5,30.10,30.39
1,Takeaway,2889,25521.5,30.28,29.94
2,Unknown,3779,33818.5,39.61,39.67


Unknown agrupa 3,779 registros (39.61%) y USD 33,818.50 (39.67% del importe). La cobertura limita cualquier comparación entre consumo en local y para llevar. Estos registros se mantienen en los totales por producto y por mes; no se redistribuyen a una modalidad conocida.

### Detalle mensual por producto
Esta tabla complementaria cumple el diseño temporal de AE2 y queda lista para gráficos de AE4. Su nivel es mes y producto; no debe sumarse junto con otras tablas de diferente nivel porque duplicaría los mismos importes.

In [10]:
display(tablas['ventas_mes_producto'].head(16))
print('Filas del detalle mes-producto:', len(tablas['ventas_mes_producto']))

Filas del detalle mes-producto: 96


,mes,producto,registros,unidades,importe_usd
0,2023-01,Cake,95,289,867.0
1,2023-01,Coffee,90,281,578.0
2,2023-01,Cookie,84,249,271.0
3,2023-01,Juice,173,527,1497.0
4,2023-01,Salad,105,314,1498.0
5,2023-01,Sandwich,112,348,1380.0
6,2023-01,Smoothie,70,191,747.0
7,2023-01,Tea,89,264,420.0
8,2023-02,Cake,75,228,684.0
9,2023-02,Coffee,85,286,580.0


## 5 · Validación y exportación automática
Se comprueba que todas las agrupaciones conserven conteos y, cuando corresponde, unidades e importes. Cada participación suma 100 antes de redondear. Se comprueban los doce meses y las 96 combinaciones mes-producto.

El CSV preparado y todas las tablas se escriben con rutas relativas y se vuelven a leer para contrastar valores. El CSV no conserva los tipos de pandas: el diccionario documenta cómo recuperarlos. El TXT se sobrescribe desde el DataFrame final e incluye dimensiones, tipos, nulos, estadísticas numéricas y temporales y frecuencias categóricas. Se compara su contenido con una nueva generación desde el mismo DataFrame. El manifiesto registra versiones y huellas SHA256.

In [11]:
display(validaciones)
manifiesto = exportar(RAIZ, original, df, calidad, comparacion, cambios,
                      tablas, validaciones, ruta_origen)
print('Validaciones de agrupaciones:', len(validaciones))
print('Archivos CSV exportados y releídos:', manifiesto['exportaciones_releidas'])
print('Resumen descriptivo: AE3/resultados/resumen_dataframe.txt')
print((RAIZ / 'AE3/resultados/resumen_dataframe.txt').read_text(encoding='utf-8'))
display(pd.DataFrame({'archivo':list(manifiesto['archivos_sha256']),
                      'sha256':list(manifiesto['archivos_sha256'].values())}))
pd.testing.assert_frame_equal(original, original_respaldo)
print('Flujo completado: original intacto, resultados conciliados y TXT regenerado.')

Validaciones de agrupaciones: 20
Archivos CSV exportados y releídos: 12
Resumen descriptivo: AE3/resultados/resumen_dataframe.txt
AE3 | Jose David Osorio Gallego
Dataset: adityasuyal/cafe-sales-cleaned
Revisión: f4e6dfa1fe791ba84807921c949bd84f02b4bc69
SHA256 CSV original: f1271f9ecffdbd4928e21b624c71ff7fe29e1eb79d3f7426e93076e49c4b30ca
Filas: 9540
Columnas: 13
Periodo: 2023-01-01 a 2023-12-31
Unidades totales: 28850
Importe total USD: 85246.50
Modalidad Unknown: 3779

ESQUEMA Y NULOS
                                  tipo  nulos  porcentaje_nulos
id_transaccion                  string      0               0.0
producto                      category      0               0.0
cantidad                         Int64      0               0.0
precio_unitario_usd            float64      0               0.0
importe_usd                    float64      0               0.0
medio_pago                    category      0               0.0
modalidad                     category      0               0.

,validacion,resultado
0,p1_unidades_producto conserva registros,APROBADA
1,p1_unidades_producto conserva unidades,APROBADA
2,p1_unidades_producto participacion_unidades_pct suma 100,APROBADA
3,p2_importe_producto conserva registros,APROBADA
4,p2_importe_producto conserva total USD,APROBADA
5,p2_importe_producto participacion_importe_pct suma 100,APROBADA
6,p3_ventas_mes conserva registros,APROBADA
7,p3_ventas_mes conserva total USD,APROBADA
8,p3_ventas_mes conserva unidades,APROBADA
9,p3_ventas_mes participacion_importe_pct suma 100,APROBADA


,archivo,sha256
0,AE3/datos/preparados/cafe_2023_preparado.csv,2112747639e2cfe028cf65c79ae1f992420cb516ca79dd79b7c1ae776bb4a3d0
1,AE3/resultados/comparacion_esquema.csv,57d20b10d191d20984085354158aacc5bbf023003bb705d8d199b8a40c41df25
2,AE3/resultados/controles_calidad.csv,a4ef612961c4709199722460c32808f9b3a2d6c4b11028792dbba31d92966498
3,AE3/resultados/diccionario_datos.csv,6f34b63f4891680b42957327b0bc4116da935b0a34d0cb97fc28500906dd23bb
4,AE3/resultados/matriz_preguntas.csv,43c4bef4386df59727c48727619874d698e83d1f7604027b7358da2e9161e34a
5,AE3/resultados/p1_unidades_producto.csv,bd5e245677258eacacf893c72eee247f0fe9b12f9c65337154987df410ee9422
6,AE3/resultados/p2_importe_producto.csv,4790f03b89ca0a32b2db4b2bcc124a48b5f01748bdcd8655416bde5cafc2654b
7,AE3/resultados/p3_ventas_mes.csv,5bba72a014f56d26cd3269978f6796c421b8ef8a07eeddf23e79b8dab0a18aff
8,AE3/resultados/p4_cobertura_modalidad.csv,03b3f40b03c26574086be79a287ac1c63f8f0e817a2a9f6d2283d1852209cf7a
9,AE3/resultados/registro_transformaciones.csv,ce2c6eb26a8e5a84a731b0f49cea15832cfd9bac3993f98135e7850979b186f8


## 6 · Conclusiones del avance y límites
Las tablas responden la pregunta central de AE2 mediante sumas y participaciones por producto, y desarrollan el complemento mensual previsto. La modalidad desconocida se cuantifica sin excluirla de los totales. Los resultados quedan preparados para representarse gráficamente en AE4.

La ausencia de nulos de pandas no implica ausencia de incertidumbre: hay `Unknown` y valores imputados previamente por el publicador. No se pueden identificar esas imputaciones fila por fila, restaurar las 460 filas excluidas, medir rentabilidad ni generalizar a una cafetería real. Un único año sintético no acredita estacionalidad ni causalidad.

## Referencias
- Suyal, A. (s. f.). *Cafe Sales Dataset (Cleaned)*. [Ficha y revisión de la fuente](https://huggingface.co/datasets/adityasuyal/cafe-sales-cleaned/tree/f4e6dfa1fe791ba84807921c949bd84f02b4bc69).
- pandas. (s. f.). [DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html), [to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html), [groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html).
- Osorio Gallego, J. D. (2026). *AE2 Ideación, diseño y documentación de una necesidad*. Antecedente conservado en `AE2/`.

Las estadísticas del avance se obtienen de esta ejecución; la ficha fuente fundamenta procedencia, moneda, licencia y tratamiento previo.